In [ ]:
import duckdb
import hashlib
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

con = duckdb.connect()

con.execute("""CREATE OR REPLACE TABLE titanic AS SELECT * FROM read_csv_auto('/home/oleg/projects/personal_projects/dataanalysis_mcp/fixtures/titanic.csv', SAMPLE_SIZE=-1)""")
titanic_df = con.sql("SELECT * FROM titanic").df()

# --- Re-fit model 'survival_logit' (kind=logistic) ---
expected_hash_survival_logit = '7b9054553579934e061e3125097e4e473eabba3ec6b9e1afbc1d1ee684e462bc'
actual_hash_survival_logit = hashlib.sha256(open('/home/oleg/projects/personal_projects/dataanalysis_mcp/fixtures/titanic.csv', 'rb').read()).hexdigest()
assert actual_hash_survival_logit == expected_hash_survival_logit, "Training data for 'survival_logit' changed since the session was recorded."
survival_logit = smf.logit("Survived ~ C(Sex) + C(Pclass) + Age + Q("Siblings/Spouses Aboard") + Q("Parents/Children Aboard") + Fare", data=titanic_df).fit(disp=False)

### Loaded dataset `titanic`
- Source: `/home/oleg/projects/personal_projects/dataanalysis_mcp/fixtures/titanic.csv`
- 887 rows x 8 columns

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE titanic AS
    SELECT * FROM read_csv_auto('/home/oleg/projects/personal_projects/dataanalysis_mcp/fixtures/titanic.csv', SAMPLE_SIZE=-1)
""")
titanic_df = con.sql("SELECT * FROM titanic").df()
titanic_df.head()

### Profiled `titanic`
- 887 rows x 8 columns
- Column `Sex` looks categorical — try compare_groups across it.
- Numeric column `Survived` is available — describe_column would surface outliers.

In [ ]:
profile_df = con.sql("SELECT * FROM titanic").df()
profile_df.describe(include='all')

### Correlation matrix on `titanic` (pearson)
- Columns: Survived, Pclass, Age, Siblings/Spouses Aboard, Parents/Children Aboard, Fare

In [ ]:
from scipy import stats
df = con.sql('SELECT "Survived", "Pclass", "Age", "Siblings/Spouses Aboard", "Parents/Children Aboard", "Fare" FROM titanic').df()
df.corr(method='pearson')

### Compared `Survived` across `Sex` groups
- Test selected: **mann_whitney**
- statistic = 40319.5000, p_value = 0.0000
- effect_size (rank_biserial) = 0.5518
- Groups `male` and `female` differ significantly (Mann-Whitney U, p=0.0000, rank_biserial=0.552).

In [ ]:
from scipy import stats
df = con.sql('SELECT * FROM titanic').df()
# Test selected automatically by compare_groups: mann_whitney


### Compared `Survived` across `Pclass` groups
- Test selected: **kruskal_wallis**
- statistic = 101.1026, p_value = 0.0000
- effect_size (epsilon_squared) = 0.1141
- At least one of ['1', '2', '3'] differs significantly (Kruskal-Wallis, p=0.0000, epsilon_squared=0.114).

In [ ]:
from scipy import stats
df = con.sql('SELECT * FROM titanic').df()
# Test selected automatically by compare_groups: kruskal_wallis


### Compared `Age` across `Survived` groups
- Test selected: **mann_whitney**
- statistic = 96539.5000, p_value = 0.3677
- effect_size (rank_biserial) = -0.0359
- No statistically significant difference between `0` and `1` at α=0.05 (Mann-Whitney U, p=0.3677, rank_biserial=-0.036).

In [ ]:
from scipy import stats
df = con.sql('SELECT * FROM titanic').df()
# Test selected automatically by compare_groups: mann_whitney


### Compared `Fare` across `Survived` groups
- Test selected: **mann_whitney**
- statistic = 57578.0000, p_value = 0.0000
- effect_size (rank_biserial) = 0.3822
- Groups `0` and `1` differ significantly (Mann-Whitney U, p=0.0000, rank_biserial=0.382).

In [ ]:
from scipy import stats
df = con.sql('SELECT * FROM titanic').df()
# Test selected automatically by compare_groups: mann_whitney


### Compared `Fare` across `Pclass` groups
- Test selected: **kruskal_wallis**
- statistic = 436.4272, p_value = 0.0000
- effect_size (epsilon_squared) = 0.4926
- At least one of ['1', '2', '3'] differs significantly (Kruskal-Wallis, p=0.0000, epsilon_squared=0.493).

In [ ]:
from scipy import stats
df = con.sql('SELECT * FROM titanic').df()
# Test selected automatically by compare_groups: kruskal_wallis


### Query

```
SELECT Sex, Pclass, COUNT(*) AS n, ROUND(AVG(Survived) * 100, 1) AS survival_pct FROM titanic GROUP BY Sex, Pclass ORDER BY Sex, Pclass
```

- 6 rows returned

In [ ]:
con.sql('SELECT Sex, Pclass, COUNT(*) AS n, ROUND(AVG(Survived) * 100, 1) AS survival_pct FROM titanic GROUP BY Sex, Pclass ORDER BY Sex, Pclass LIMIT 50').df()

### Query

```
SELECT CASE WHEN Age < 13 THEN 'child (<13)' WHEN Age < 18 THEN 'teen (13-17)' WHEN Age < 35 THEN 'young adult (18-34)' WHEN Age < 55 THEN 'middle (35-54)' ELSE 'senior (55+)' END AS age_band, COUNT(*) AS n, ROUND(AVG(Survived) * 100, 1) AS survival_pct FROM titanic GROUP BY age_band ORDER BY MIN(Age)
```

- 5 rows returned

In [ ]:
con.sql("SELECT CASE WHEN Age < 13 THEN 'child (<13)' WHEN Age < 18 THEN 'teen (13-17)' WHEN Age < 35 THEN 'young adult (18-34)' WHEN Age < 55 THEN 'middle (35-54)' ELSE 'senior (55+)' END AS age_band, COUNT(*) AS n, ROUND(AVG(Survived) * 100, 1) AS survival_pct FROM titanic GROUP BY age_band ORDER BY MIN(Age) LIMIT 50").df()

### Query

```
SELECT ("Siblings/Spouses Aboard" + "Parents/Children Aboard") AS family_size, COUNT(*) AS n, ROUND(AVG(Survived) * 100, 1) AS survival_pct FROM titanic GROUP BY family_size ORDER BY family_size
```

- 9 rows returned

In [ ]:
con.sql('SELECT ("Siblings/Spouses Aboard" + "Parents/Children Aboard") AS family_size, COUNT(*) AS n, ROUND(AVG(Survived) * 100, 1) AS survival_pct FROM titanic GROUP BY family_size ORDER BY family_size LIMIT 50').df()

### Fitted LOGISTIC model on `titanic`
- Formula: `Survived ~ C(Sex) + C(Pclass) + Age + Q("Siblings/Spouses Aboard") + Q("Parents/Children Aboard") + Fare`
- Strongest predictor: `C(Sex)[T.male]` (negative effect, statistically significant at α=0.05, p=5.895e-43). A one-unit increase changes the odds by ~-93.6% (odds ratio = 0.064).
- pseudo-R² = 0.3397
- AIC = 796.93, BIC = 835.23

In [ ]:
import statsmodels.formula.api as smf
df = con.sql("SELECT * FROM titanic").df()
model = smf.logit("Survived ~ C(Sex) + C(Pclass) + Age + Q("Siblings/Spouses Aboard") + Q("Parents/Children Aboard") + Fare", data=df).fit(disp=0)
model.summary()

### Evaluated `survival_logit` on `titanic`
- ROC-AUC = 0.857, PR-AUC = 0.827, Brier = 0.139
- Accuracy = 0.803, F1 = 0.732 at threshold 0.5
- Calibration: 10 bins, max decile gap = 0.088

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
y_true = titanic_df["Survived"]
y_pred = survival_logit.predict(titanic_df)
print(f"ROC-AUC = {roc_auc_score(y_true, y_pred):.3f}")
print(f"PR-AUC  = {average_precision_score(y_true, y_pred):.3f}")
print(f"Brier   = {brier_score_loss(y_true, y_pred):.3f}")
# Calibration table (quantile bins):
_q = pd.qcut(y_pred, q=10, duplicates='drop', labels=False)
calibration = pd.DataFrame({'bin': _q + 1, 'y_true': y_true, 'y_pred': y_pred}).groupby('bin').agg(
    mean_predicted=('y_pred', 'mean'), mean_observed=('y_true', 'mean'), n=('y_true', 'size')
).reset_index()
calibration

### Plotted `bar` of dataset `titanic`
- x: `Pclass`
- y: `Survived`
- hue: `Sex`
- title: 'Survival rate by class and sex'

In [ ]:
import matplotlib.pyplot as plt
_df = con.sql("SELECT Pclass, Survived, Sex FROM titanic").df()
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
_means = _df.groupby('Pclass')['Survived'].mean()
ax.bar(_means.index.astype(str), _means.values)
ax.set_xlabel('Pclass')
ax.set_ylabel('avg(Survived)')
ax.set_title('Survival rate by class and sex')
plt.show()

### Plotted `violin` of dataset `titanic`
- x: `Survived`
- y: `Age`
- title: 'Age distribution by survival'

In [ ]:
import matplotlib.pyplot as plt
_df = con.sql("SELECT Survived, Age FROM titanic").df()
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
_groups = _df.groupby('Survived')['Age'].apply(list)
ax.violinplot(_groups.values)
ax.set_xticks(range(1, len(_groups) + 1))
ax.set_xticklabels(_groups.index.astype(str).tolist())
ax.set_xlabel('Survived')
ax.set_ylabel('Age')
ax.set_title('Age distribution by survival')
plt.show()

### Plotted `box` of dataset `titanic`
- x: `Pclass`
- y: `Fare`
- title: 'Fare distribution by class'

In [ ]:
import matplotlib.pyplot as plt
_df = con.sql("SELECT Pclass, Fare FROM titanic").df()
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
_groups = _df.groupby('Pclass')['Fare'].apply(list)
ax.boxplot(_groups.values, tick_labels=_groups.index.astype(str).tolist())
ax.set_xlabel('Pclass')
ax.set_ylabel('Fare')
ax.set_title('Fare distribution by class')
plt.show()

### Plotted `heatmap` of dataset `titanic`
- title: 'Correlation heatmap (Pearson)'

In [ ]:
import matplotlib.pyplot as plt
_df = con.sql("SELECT * FROM titanic").df()
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
_corr = con.sql('SELECT * FROM titanic').df().corr(numeric_only=True)
im = ax.imshow(_corr.values, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(_corr.columns)))
ax.set_yticks(range(len(_corr.columns)))
ax.set_xticklabels(_corr.columns, rotation=45, ha='right')
ax.set_yticklabels(_corr.columns)
fig.colorbar(im, ax=ax)
ax.set_title('Correlation heatmap (Pearson)')
plt.show()